In [1]:
from plateaukit import load_dataset
import pandas as pd
import ipywidgets as widgets
from plateaukit.core.widgets.interactive_deck import InteractiveDeck

# データ読み込み
dataset = load_dataset("plateau-23100-nagoya-shi-2022")
area = dataset.area_from_landmark("名古屋駅")

# 浸水深(depth)を取り出す
depth = area.gdf["riverFloodingRisk"].apply(
    lambda x: x["null"]["depth"]
    if isinstance(x, dict)
    and "null" in x
    and isinstance(x["null"], dict)
    and "depth" in x["null"]
    else None
)

area.gdf["riverFloodingRiskDepthCm"] = depth * 100  # m → cm
# print(area.gdf["riverFloodingRiskDepthCm"].describe())  #debug

def show_widget():
    # 浸水深が定義されていない建物を最初から灰色にする
    initial_gdf = area.gdf.copy()

    initial_gdf["fill_color"] = initial_gdf[
        "riverFloodingRiskDepthCm"
    ].isna().map({
        True: [100, 100, 100],
        False: None
    })

    # タイトル
    title = widgets.HTML(
        value="<h2>浸水深シミュレーション</h2>"
    )

    # このデータで地図を作成
    deck = InteractiveDeck(initial_gdf)

    # スライダー
    slider = widgets.FloatSlider(
        value=0,
        min=0,
        max=200,
        step=1,
        description="浸水深(cm)",
        layout=widgets.Layout(width="100%"),
        readout_format=".0f",
    )

    output = widgets.Output()

    with output:
        display(deck.deck)

    def handler(change):
        water_level = change["new"]

        gdf = area.gdf.copy()

        # 浸水深が定義されているか
        defined = gdf["riverFloodingRiskDepthCm"].notna()

        # 指定した水深以上の建物
        flooded = (
            defined
            & (gdf["riverFloodingRiskDepthCm"] <= water_level)
        )

        # 色を設定
        gdf["fill_color"] = [
            [0, 0, 255] if f else
            [100, 100, 100] if not d else
            None
            for f, d in zip(flooded, defined)
        ]

        updated_layer = deck._create_building_layer(gdf)
        deck.deck.layers = [updated_layer]
        deck.deck.update()

    # スライダーを動かしたときにだけ更新
    slider.observe(handler, names="value")

    display(title, slider, output)

show_widget()

HTML(value='<h2>浸水深シミュレーション</h2>')

FloatSlider(value=0.0, description='浸水深(cm)', layout=Layout(width='100%'), max=200.0, readout_format='.0f', st…

Output()